# Optional · Upper limits with CL$_s$ (RooStats)
When the data show no excess, we quote an **upper limit**. Model: on/off with a Gaussian constraint (Lab 3).
Use case: a counting search (CUORE-0 peak, dark-matter, rare decay).
Calculator: `AsymptoticCalculator` + `HypoTestInverter`, method CL$_s$ at 95 %.

In [ ]:
import ROOT
import numpy as np
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
def build(n_obs, b_hat, sigma_b):
    w = ROOT.RooWorkspace("w")
    s = ROOT.RooRealVar("s", "signal", 1, 0, 30)
    b = ROOT.RooRealVar("b", "background", b_hat, 0.01, 30)
    b0 = ROOT.RooRealVar("b0", "b0", b_hat, 0, 30); b0.setConstant(True)
    sb = ROOT.RooRealVar("sb", "sb", sigma_b); sb.setConstant(True)
    n = ROOT.RooRealVar("n", "n", 0, 100)
    mu_n = ROOT.RooFormulaVar("mu_n", "@0+@1", ROOT.RooArgList(s, b))
    pois = ROOT.RooPoisson("pois", "pois", n, mu_n)
    cons = ROOT.RooGaussian("cons", "cons", b0, b, sb)
    model = ROOT.RooProdPdf("model", "model", ROOT.RooArgList(pois, cons))
    data = ROOT.RooDataSet("data", "data", ROOT.RooArgSet(n)); n.setVal(n_obs); data.add(ROOT.RooArgSet(n))
    w.Import(model); w.Import(data)
    mc = ROOT.RooStats.ModelConfig("mc_sb", w)
    mc.SetPdf(w.pdf("model")); mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("s")))
    mc.SetNuisanceParameters(ROOT.RooArgSet(w.var("b"))); mc.SetObservables(ROOT.RooArgSet(w.var("n")))
    mc.SetGlobalObservables(ROOT.RooArgSet(w.var("b0")))
    w.var("s").setVal(1); mc.SetSnapshot(ROOT.RooArgSet(w.var("s")))
    mc_b = mc.Clone("mc_b")
    w.var("s").setVal(0); mc_b.SetSnapshot(ROOT.RooArgSet(w.var("s")))
    return w, mc, mc_b

def upper_limit(n_obs, b_hat, sigma_b, cl=0.95):
    w, mc, mc_b = build(n_obs, b_hat, sigma_b)
    ac = ROOT.RooStats.AsymptoticCalculator(w.data("data"), mc, mc_b)
    ac.SetOneSided(True)
    inv = ROOT.RooStats.HypoTestInverter(ac)
    inv.SetConfidenceLevel(cl); inv.UseCLs(True); inv.SetFixedScan(40, 0.0, 20.0)
    r = inv.GetInterval()
    return r.UpperLimit(), r.GetExpectedUpperLimit(0), (w, mc, mc_b, ac, inv, r)

ul, ul_exp, keep = upper_limit(n_obs=3, b_hat=3.8, sigma_b=0.6)
print("n_obs = 3, b = 3.8 +- 0.6:   observed limit = %.2f   expected (median) = %.2f" % (ul, ul_exp))

In [ ]:
check("limit is positive", ul > 0)
check("observed limit is below the median expectation when n_obs < b (a downward fluctuation)", ul < ul_exp * 1.2)

## Questions
1. Run `n_obs = 3, 4, 8`. How do the observed and the expected limits move? Why can the observed limit be **lower** than the expected one?
2. Why do we use CL$_s$ instead of the plain CL$_{s+b}$ when there is a downward fluctuation?
3. Plot `inv.GetInterval()` with `RooStats.HypoTestInverterPlot`. What are the green and yellow bands?
4. Replace the Gaussian constraint with the Poisson auxiliary measurement ($n_{\\rm off}=4$, $\\tau=1.053$). How does the limit change?